# F1 Dataset Guide: Coverage, Quality and Joins

A practical tour of the canonical Formula 1 tables: what each row means, which seasons are covered, how quality decisions are documented, and how to join the files safely.

**You will learn:** table grain, historical availability boundaries, key integrity, and modeling-time leakage boundaries. Missing telemetry before its source era means *unavailable*, not zero.


In [ ]:
import os
from pathlib import Path
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

warnings.filterwarnings("ignore", category=FutureWarning)
sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", 100)

DATA_DIR = Path(os.getenv("F1_DATA_DIR", "/kaggle/input/formula-1-pit-stop-dataset"))
if not DATA_DIR.exists():
    raise FileNotFoundError(f"Dataset directory not found: {DATA_DIR}")
print(f"Reading data from {DATA_DIR}")


In [ ]:
files = ["race_context", "race_drivers", "pit_events", "stints", "weather_observations"]
tables = {name: pd.read_csv(DATA_DIR / f"{name}.csv", low_memory=False) for name in files}
coverage = pd.read_csv(DATA_DIR / "coverage.csv")
issues = pd.read_csv(DATA_DIR / "data_quality_issues.csv", low_memory=False)
dictionary = pd.read_csv(DATA_DIR / "data_dictionary.csv")

inventory = pd.DataFrame({
    "table": files,
    "rows": [len(tables[n]) for n in files],
    "columns": [tables[n].shape[1] for n in files],
    "earliest_season": [tables[n]["season"].min() for n in files],
    "latest_season": [tables[n]["season"].max() for n in files],
})
inventory


## Historical coverage

The tables deliberately begin in different years because source availability differs. This prevents false zeroes in early seasons.


In [ ]:
display(coverage)
ax = coverage.sort_values("earliest_season").plot.barh(
    x="table", y="row_count", figsize=(10, 4), legend=False, color="#e10600"
)
ax.set(title="Published rows by table", xlabel="Rows", ylabel="")
plt.tight_layout()


## Key integrity and join map

Use `(season, round_number)` for a race, add `driver_id` for a driver-race, and use `session_key` where modern session-level data provides it.


In [ ]:
checks = {
    "race_context unique race": ~tables["race_context"].duplicated(["season", "round_number"]).any(),
    "race_drivers unique entry": ~tables["race_drivers"].duplicated(["season", "round_number", "driver_id", "car_number"]).any(),
    "pit_events unique stop": ~tables["pit_events"].duplicated(["season", "round_number", "driver_id", "stop_number"]).any(),
    "stints unique stint": ~tables["stints"].duplicated(["season", "round_number", "driver_id", "stint_number"]).any(),
    "all driver races have context": tables["race_drivers"].merge(
        tables["race_context"][["season", "round_number"]].drop_duplicates(),
        on=["season", "round_number"], how="left", indicator=True
    )["_merge"].eq("both").all(),
}
pd.Series(checks, name="passed").to_frame()


## Missingness is meaningful

The chart below highlights columns whose availability is tied to a source era. Read `coverage.csv` and the dictionary before imputing values.


In [ ]:
missing = pd.concat({name: frame.isna().mean() for name, frame in tables.items()}).rename("missing_rate")
missing = missing[missing.gt(0)].sort_values(ascending=False).head(25).reset_index()
missing.columns = ["table", "column", "missing_rate"]
display(missing)
plt.figure(figsize=(9, 7))
sns.barplot(data=missing, y=missing["table"] + "." + missing["column"], x="missing_rate", color="#3671c6")
plt.title("Largest documented missingness rates")
plt.xlabel("Fraction missing")
plt.ylabel("")
plt.tight_layout()


## Quality ledger

Every normalization or coverage decision is published. `warning` is included but documented; an error would exclude the affected table for that race.


In [ ]:
display(issues.groupby(["severity", "resolution"], dropna=False).size().rename("issues").reset_index())
issues["issue_code"].value_counts().head(12).to_frame("count")


## Modeling checklist

- Define the prediction moment first.
- Use only fields available by that moment.
- Split chronologically by race or season.
- Do not interpret absent pre-2011 pit rows as zero stops.
- Treat 2026 as a partial season until it is complete.
- Keep targets such as `classified_position` out of predictors.
